# Benchmark: serial vs. parallel execution of vectorized_pipeline functions

For each metric function and each **characteristic length** (expected stops per user),
this notebook measures:

* **Serial** execution time — one process, all 2 000 users in a single dataframe.
* **Parallel** execution time — `ProcessPoolExecutor` with the dataframe split into
  chunks of `batch_size` users each.

Outputs
-------
```
serial_df   columns: [time_execution, function_executed, characteristic_length]
parallel_df columns: [time_execution, batch_size, function_executed, characteristic_length]
```
The final **analysis** cell determines which regime (serial or a specific batch_size)
minimises wall-clock time for every parameter combination.

In [ ]:
# ---------------------------------------------------------------------------
# Imports and project path setup
# ---------------------------------------------------------------------------
import os
import sys
import time
import math
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Make the project root importable regardless of where the notebook is opened
PROJECT_ROOT = Path().resolve().parent
TESTS_DIR    = Path().resolve()
for _p in [str(PROJECT_ROOT), str(TESTS_DIR)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import geohash as gh  # available in the project venv

from src.vectorized_pipeline import (
    _compute_radius_of_gyration_polars,
    _compute_krg_polars,
    _compute_entropies_polars,
    _compute_real_entropy_polars,
    _compute_home_polars,
    _compute_distance_polars,
    _compute_st_polars,
    _compute_gonzalez_polars,
    _compute_frequency_polars,
    _compute_weekly_rg_polars,
)

from bench_workers import (
    rg_worker,
    krg_worker,
    entropies_worker,
    home_worker,
    distance_worker,
    weekly_rg_worker,
    _re_worker,
    _st_worker,
    _gonz_worker,
    _freq_worker,
    K_VALUES,
    T_THRESHOLD,
    PERIOD_DIVISION,
    PERIOD_NAME,
    PERODNAME2IDX,
)

print("Polars version :", pl.__version__)
print("Project root   :", PROJECT_ROOT)

## 1. Constants

In [ ]:
# ---------------------------------------------------------------------------
# Benchmark parameters
# ---------------------------------------------------------------------------
N_USERS               = 2_000
CHARACTERISTIC_LENGTHS = [20, 50, 100, 500, 700]
BATCH_SIZES           = [100, 500, 1_000, 2_000]

# Detected or capped CPU count
MAX_WORKERS = min(4, os.cpu_count() or 4)
print(f"MAX_WORKERS = {MAX_WORKERS}")

# Geographic bounds (California bounding box)
LAT_MIN, LAT_MAX = 32.5, 42.0
LON_MIN, LON_MAX = -124.5, -114.0

# Synthetic data seed
SEED = 2024

# Base datetime for trajectory timestamps
BASE_DATE = datetime(2020, 1, 1)

## 2. Synthetic trajectory generator

Each of the 2 000 users gets a trajectory whose **length** (number of GPS stops)
is drawn from `Exponential(characteristic_length)`, with a minimum of 3.

Stops are drawn from a **pool of 200 unique locations** inside the California
bounding box; each user has their own random subset of 5-15 locations with
Pareto-like visit probabilities — so the data is realistic enough for entropy
and frequency metrics.

Timestamps are **serial** (one stop per hour starting at 2020-01-01 00:00).

In [ ]:
def generate_trajectories(
    n_users: int,
    characteristic_length: float,
    seed: int = SEED,
) -> pl.DataFrame:
    """
    Generate a synthetic trajectory dataset.

    Parameters
    ----------
    n_users : int
        Number of distinct users.
    characteristic_length : float
        Expected number of GPS stops per user (exponential parameter).
    seed : int
        NumPy random seed (modified internally per characteristic_length).

    Returns
    -------
    pl.DataFrame
        Columns: [userId, lat, lon, begin, end, dur_min, geohash7]
    """
    rng = np.random.default_rng(seed + int(characteristic_length * 100))

    # ------------------------------------------------------------------
    # Build a shared location pool (200 points inside CA bbox)
    # ------------------------------------------------------------------
    N_POOL = 200
    pool_lats = rng.uniform(LAT_MIN, LAT_MAX, N_POOL)
    pool_lons = rng.uniform(LON_MIN, LON_MAX, N_POOL)
    pool_gh7  = np.array(
        [gh.encode(lat, lon, 7) for lat, lon in zip(pool_lats, pool_lons)],
        dtype=object,
    )

    # ------------------------------------------------------------------
    # Trajectory lengths ~ Exp(characteristic_length), min 3
    # ------------------------------------------------------------------
    lengths     = np.maximum(3, rng.exponential(characteristic_length, n_users).astype(int))
    total_stops = int(lengths.sum())

    # ------------------------------------------------------------------
    # Pre-allocate output arrays
    # ------------------------------------------------------------------
    uid_col  = [None] * total_stops   # str
    lat_col  = np.empty(total_stops)
    lon_col  = np.empty(total_stops)
    begin_ms = np.empty(total_stops, dtype=np.int64)  # ms since epoch
    gh7_col  = [None] * total_stops   # str

    # 2020-01-01 00:00:00 UTC in milliseconds since Unix epoch
    t0_ms    = int(pd.Timestamp(BASE_DATE).timestamp() * 1000)
    hour_ms  = 3_600_000  # 1 h in ms

    pos = 0
    for uid in range(n_users):
        n = int(lengths[uid])

        # Personal subset of locations
        n_personal = int(rng.integers(5, min(16, N_POOL)))
        loc_idxs   = rng.choice(N_POOL, size=n_personal, replace=False)
        weights    = rng.exponential(1.0, n_personal)
        probs      = weights / weights.sum()
        chosen     = rng.choice(loc_idxs, size=n, p=probs)

        uid_col[pos:pos + n]  = [str(uid)] * n
        lat_col[pos:pos + n]  = pool_lats[chosen]
        lon_col[pos:pos + n]  = pool_lons[chosen]
        begin_ms[pos:pos + n] = t0_ms + np.arange(n, dtype=np.int64) * hour_ms
        gh7_col[pos:pos + n]  = pool_gh7[chosen].tolist()

        pos += n

    end_ms = begin_ms + hour_ms

    return pl.DataFrame({
        "userId"  : pl.Series(uid_col,  dtype=pl.Utf8),
        "lat"     : pl.Series(lat_col,  dtype=pl.Float64),
        "lon"     : pl.Series(lon_col,  dtype=pl.Float64),
        "begin"   : pl.Series(begin_ms, dtype=pl.Int64).cast(pl.Datetime("ms")),
        "end"     : pl.Series(end_ms,   dtype=pl.Int64).cast(pl.Datetime("ms")),
        "dur_min" : pl.Series(np.full(total_stops, 60.0)),
        "geohash7": pl.Series(gh7_col,  dtype=pl.Utf8),
    })

In [ ]:
# ---------------------------------------------------------------------------
# Generate one dataset per characteristic length and print summary stats
# ---------------------------------------------------------------------------
print("Generating synthetic datasets …")
t0 = time.perf_counter()
datasets: dict[int, pl.DataFrame] = {}
for cl in CHARACTERISTIC_LENGTHS:
    df = generate_trajectories(N_USERS, cl)
    datasets[cl] = df
    n_rows  = len(df)
    avg_len = n_rows / N_USERS
    print(f"  CL={cl:4d} → {n_rows:>9,} rows  (avg {avg_len:6.1f} stops/user)")
print(f"Done in {time.perf_counter()-t0:.1f}s")

## 3. Serial benchmarks

Run each metric function **once** on the full dataframe (all 2 000 users).

Output → `serial_df` with columns `[time_execution, function_executed, characteristic_length]`.

In [ ]:
def _time_it(fn, *args) -> float:
    """Return wall-clock seconds for fn(*args)."""
    t = time.perf_counter()
    fn(*args)
    return time.perf_counter() - t


def benchmark_all_serial(
    datasets: dict[int, pl.DataFrame],
) -> pd.DataFrame:
    """
    Run every metric function serially for each dataset.

    Returns a DataFrame with columns:
        [time_execution, function_executed, characteristic_length]
    """
    # Each entry: (display_name, callable, extra_positional_args)
    SERIAL_SPECS = [
        ("radius_of_gyration",   _compute_radius_of_gyration_polars, ()),
        ("krg",                  _compute_krg_polars,                (K_VALUES,)),
        ("entropies",            _compute_entropies_polars,          ()),
        ("real_entropy",         _compute_real_entropy_polars,       ()),
        ("home",                 _compute_home_polars,               ()),
        ("distance",             _compute_distance_polars,           ()),
        ("st_curve",             _compute_st_polars,                 (T_THRESHOLD,)),
        ("gonzalez",             _compute_gonzalez_polars,           ()),
        ("frequency",            _compute_frequency_polars,          ()),
        ("weekly_rg",            _compute_weekly_rg_polars,
                                  (PERIOD_DIVISION, PERIOD_NAME, PERODNAME2IDX)),
    ]

    records = []
    total   = len(CHARACTERISTIC_LENGTHS) * len(SERIAL_SPECS)
    done    = 0

    for cl in CHARACTERISTIC_LENGTHS:
        df = datasets[cl]
        for fn_name, fn, extra in SERIAL_SPECS:
            done += 1
            print(f"  [{done}/{total}] serial  CL={cl:4d}  {fn_name} …", end=" ", flush=True)
            elapsed = _time_it(fn, df, *extra)
            print(f"{elapsed:.2f}s")
            records.append({
                "time_execution"       : elapsed,
                "function_executed"    : fn_name,
                "characteristic_length": cl,
            })

    return pd.DataFrame(records)

In [ ]:
print("=" * 60)
print("SERIAL BENCHMARKS")
print("=" * 60)
serial_df = benchmark_all_serial(datasets)
print("\nserial_df:")
serial_df

## 4. Parallel benchmarks

For each metric, the full dataframe is **split into chunks of `batch_size` users**
and each chunk is processed by a separate worker in a `ProcessPoolExecutor`.

Output → `parallel_df` with columns `[time_execution, batch_size, function_executed, characteristic_length]`.

In [ ]:
def split_by_batch_size(df: pl.DataFrame, batch_size: int) -> list[pl.DataFrame]:
    """
    Split *df* into chunks of *batch_size* users each.

    Different from ``_split_df_by_users`` which takes the number of chunks;
    this function takes the chunk *size*.
    """
    all_users = df["userId"].unique(maintain_order=False).to_list()
    return [
        df.filter(pl.col("userId").is_in(all_users[i : i + batch_size]))
        for i in range(0, len(all_users), batch_size)
    ]


def _run_pool(worker_fn, chunks: list, extra_args=None) -> None:
    """
    Submit *chunks* to *worker_fn* via ProcessPoolExecutor and block until done.

    ``extra_args``: if provided, each task is submitted as
    ``worker_fn((chunk, *extra_args))`` using pool.map (the worker must accept
    a single tuple argument).
    """
    n_workers = min(len(chunks), MAX_WORKERS)
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        if extra_args is not None:
            # _st_worker signature: (chunk_df, t_threshold) packed as one tuple
            list(pool.map(worker_fn, [(c,) + extra_args for c in chunks]))
        else:
            list(pool.map(worker_fn, chunks))


def benchmark_all_parallel(
    datasets: dict[int, pl.DataFrame],
    batch_sizes: list[int],
) -> pd.DataFrame:
    """
    Run every metric function in parallel for each (dataset, batch_size) pair.

    Returns a DataFrame with columns:
        [time_execution, batch_size, function_executed, characteristic_length]
    """
    # Each entry: (display_name, worker_fn, extra_args_tuple_or_None)
    # extra_args are packed into a single tuple with the chunk for pool.map;
    # see _run_pool and _st_worker signature.
    PARALLEL_SPECS = [
        ("radius_of_gyration", rg_worker,         None),
        ("krg",               krg_worker,         None),
        ("entropies",         entropies_worker,   None),
        ("real_entropy",      _re_worker,         None),
        ("home",              home_worker,        None),
        ("distance",          distance_worker,    None),
        # _st_worker expects a (chunk_df, t_threshold) tuple
        ("st_curve",          _st_worker,         (T_THRESHOLD,)),
        ("gonzalez",          _gonz_worker,       None),
        ("frequency",         _freq_worker,       None),
        ("weekly_rg",         weekly_rg_worker,   None),
    ]

    records = []
    total   = len(CHARACTERISTIC_LENGTHS) * len(batch_sizes) * len(PARALLEL_SPECS)
    done    = 0

    for cl in CHARACTERISTIC_LENGTHS:
        df = datasets[cl]
        for bs in batch_sizes:
            chunks = split_by_batch_size(df, bs)
            for fn_name, worker_fn, extra in PARALLEL_SPECS:
                done += 1
                print(
                    f"  [{done}/{total}] parallel  CL={cl:4d}  bs={bs:5d}  {fn_name} …",
                    end=" ",
                    flush=True,
                )
                t = time.perf_counter()
                _run_pool(worker_fn, chunks, extra)
                elapsed = time.perf_counter() - t
                print(f"{elapsed:.2f}s")
                records.append({
                    "time_execution"       : elapsed,
                    "batch_size"           : bs,
                    "function_executed"    : fn_name,
                    "characteristic_length": cl,
                })

    return pd.DataFrame(records)

In [ ]:
print("=" * 60)
print("PARALLEL BENCHMARKS")
print("=" * 60)
parallel_df = benchmark_all_parallel(datasets, BATCH_SIZES)
print("\nparallel_df:")
parallel_df

## 5. Analysis — which regime is fastest?

For every `(function_executed, characteristic_length)` combination, we compare:
* serial time
* best parallel time (over all batch sizes)

and report the winner, the speedup, and the optimal batch size.

In [ ]:
def analyze_best_regime(
    serial_df: pd.DataFrame,
    parallel_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each (function_executed, characteristic_length) pair, find the
    configuration that minimises wall-clock time.

    Returns a DataFrame with columns:
        function_executed, characteristic_length,
        serial_time, best_parallel_time, best_batch_size,
        speedup, winner
    """
    # Best parallel time per (function, characteristic_length)
    best_par = (
        parallel_df
        .sort_values("time_execution")
        .groupby(["function_executed", "characteristic_length"], sort=False)
        .first()
        .reset_index()
        .rename(columns={
            "time_execution": "best_parallel_time",
            "batch_size"    : "best_batch_size",
        })
    )

    merged = serial_df.rename(columns={"time_execution": "serial_time"}).merge(
        best_par[["function_executed", "characteristic_length",
                  "best_parallel_time", "best_batch_size"]],
        on=["function_executed", "characteristic_length"],
        how="left",
    )

    merged["speedup"] = (merged["serial_time"] / merged["best_parallel_time"]).round(2)
    merged["winner"]  = merged.apply(
        lambda r: "parallel" if r["best_parallel_time"] < r["serial_time"] else "serial",
        axis=1,
    )

    return merged.sort_values(
        ["function_executed", "characteristic_length"]
    ).reset_index(drop=True)


analysis_df = analyze_best_regime(serial_df, parallel_df)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 60)
analysis_df

## 6. Visualisation

In [ ]:
# ---------------------------------------------------------------------------
# Figure 1 — Serial execution time per function × characteristic length
# ---------------------------------------------------------------------------
fn_names = serial_df["function_executed"].unique().tolist()
fn_names.sort()

fig, axes = plt.subplots(
    2, 5, figsize=(20, 6), sharey=False, constrained_layout=True
)
fig.suptitle(
    f"Serial execution time  (N_users={N_USERS})",
    fontsize=13, fontweight="bold",
)

for ax, fn in zip(axes.flat, fn_names):
    sub = serial_df[serial_df["function_executed"] == fn]
    ax.bar(
        [str(c) for c in sub["characteristic_length"]],
        sub["time_execution"],
        color="steelblue",
    )
    ax.set_title(fn, fontsize=9)
    ax.set_xlabel("characteristic_length")
    ax.set_ylabel("seconds")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))

plt.savefig("serial_times.png", dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 2 — Speedup heatmap  (serial / best_parallel)
#             rows = function, columns = characteristic_length
# ---------------------------------------------------------------------------
pivot = analysis_df.pivot(
    index="function_executed",
    columns="characteristic_length",
    values="speedup",
)

fig2, ax2 = plt.subplots(figsize=(10, 6), constrained_layout=True)
im = ax2.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=4.0)
fig2.colorbar(im, ax=ax2, label="speedup  (serial / best_parallel)")
ax2.set_xticks(range(len(pivot.columns)))
ax2.set_xticklabels(pivot.columns)
ax2.set_yticks(range(len(pivot.index)))
ax2.set_yticklabels(pivot.index)
ax2.set_xlabel("characteristic_length")
ax2.set_ylabel("function")
ax2.set_title(
    f"Speedup: serial / best-parallel  (green > 1 → parallel wins)\n"
    f"MAX_WORKERS={MAX_WORKERS}, BATCH_SIZES={BATCH_SIZES}",
    fontsize=10,
)

# Annotate cells with (speedup, best_batch_size)
for i, fn in enumerate(pivot.index):
    for j, cl in enumerate(pivot.columns):
        row = analysis_df[
            (analysis_df["function_executed"] == fn)
            & (analysis_df["characteristic_length"] == cl)
        ]
        if row.empty:
            continue
        sp  = row["speedup"].iloc[0]
        bs  = row["best_batch_size"].iloc[0]
        win = row["winner"].iloc[0]
        label = f"{sp:.1f}x\nbs={int(bs)}" if win == "parallel" else f"{sp:.1f}x\nserial"
        ax2.text(
            j, i, label,
            ha="center", va="center",
            fontsize=7,
            color="black" if 0.8 < sp < 3.0 else "white",
        )

plt.savefig("speedup_heatmap.png", dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Figure 3 — Parallel time vs. batch_size for the Python-heavy functions
# ---------------------------------------------------------------------------
PYTHON_HEAVY = ["real_entropy", "st_curve", "gonzalez", "frequency"]

fig3, axes3 = plt.subplots(
    len(PYTHON_HEAVY), len(CHARACTERISTIC_LENGTHS),
    figsize=(18, 12),
    sharey="row",
    constrained_layout=True,
)
fig3.suptitle(
    "Parallel time vs. batch_size for Python-heavy functions",
    fontsize=12, fontweight="bold",
)

for row_i, fn in enumerate(PYTHON_HEAVY):
    for col_i, cl in enumerate(CHARACTERISTIC_LENGTHS):
        ax = axes3[row_i, col_i]
        sub = parallel_df[
            (parallel_df["function_executed"] == fn)
            & (parallel_df["characteristic_length"] == cl)
        ].sort_values("batch_size")

        # Serial reference line
        ser_t = serial_df[
            (serial_df["function_executed"] == fn)
            & (serial_df["characteristic_length"] == cl)
        ]["time_execution"].iloc[0]

        ax.plot(
            [str(b) for b in sub["batch_size"]],
            sub["time_execution"],
            marker="o",
            color="tomato",
            label="parallel",
        )
        ax.axhline(ser_t, color="steelblue", linestyle="--", lw=1.5, label="serial")

        if col_i == 0:
            ax.set_ylabel(fn, fontsize=8)
        if row_i == 0:
            ax.set_title(f"CL={cl}", fontsize=9)
        if row_i == len(PYTHON_HEAVY) - 1:
            ax.set_xlabel("batch_size", fontsize=8)
        if row_i == 0 and col_i == len(CHARACTERISTIC_LENGTHS) - 1:
            ax.legend(fontsize=7)

plt.savefig("parallel_vs_serial_python_heavy.png", dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Summary table: winner and recommendation per function
# ---------------------------------------------------------------------------
summary = (
    analysis_df
    .groupby("function_executed")
    .agg(
        parallel_wins=("winner", lambda s: (s == "parallel").sum()),
        total         =("winner", "count"),
        mean_speedup  =("speedup", "mean"),
        max_speedup   =("speedup", "max"),
    )
    .reset_index()
)
summary["recommendation"] = summary.apply(
    lambda r: "parallel" if r["parallel_wins"] > r["total"] / 2 else "serial",
    axis=1,
)
summary.sort_values("mean_speedup", ascending=False).reset_index(drop=True)